# 🚗 Waymo Open Dataset — Perception Pipeline

**End-to-end autonomous driving perception pipeline built on the Waymo Open Dataset.**

This notebook covers:
1. Secure cloud data access (Google Cloud Storage)
2. Parsing raw sensor data from TFRecord format
3. Camera image visualization with ground-truth bounding boxes
4. Multi-object tracking using Waymo object IDs
5. Trajectory extraction and motion prediction modeling

**Dataset:** [Waymo Open Dataset v1.4.2](https://waymo.com/open/)  
**Storage:** `gs://waymo_open_dataset_v_1_4_2/`

---
## 1. Setup & Imports

In [ ]:
# Install required packages
# Note: Run this cell first, then restart the runtime before continuing
!pip install protobuf==3.20.3 -q
!pip install waymo-open-dataset-tf-2-11-0 --no-deps -q
print("Installation complete — restart runtime if this is your first run.")

In [ ]:
# Core imports
import os
import math
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.animation as animation
from IPython.display import HTML

import tensorflow as tf
from waymo_open_dataset import dataset_pb2

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print(f"TensorFlow: {tf.__version__}")
print("All imports successful.")

---
## 2. Authentication & Data Access

We authenticate with Google Cloud to access the Waymo dataset bucket directly.  
No local download needed — data is streamed from GCS.

In [ ]:
# Authenticate Google Colab with your Google account
from google.colab import auth
auth.authenticate_user()
print("Authentication successful.")

In [ ]:
# Download a single segment for local parsing
# Waymo requires accepting their terms at https://waymo.com/open/
SEGMENT_NAME = "segment-10017090168044687777_6380_000_6400_000_with_camera_labels.tfrecord"
GCS_PATH = f"gs://waymo_open_dataset_v_1_4_2/individual_files/training/{SEGMENT_NAME}"

if not os.path.exists(SEGMENT_NAME):
    print("Downloading segment...")
    !gsutil cp {GCS_PATH} .
    print("Download complete.")
else:
    print("Segment already downloaded.")

---
## 3. Data Loading & Parsing

Waymo stores data as TFRecords — a binary format containing serialized protobuf messages.  
Each **frame** contains:
- 5 camera images (FRONT, FRONT_LEFT, FRONT_RIGHT, SIDE_LEFT, SIDE_RIGHT)
- LiDAR point clouds
- Ground-truth bounding box annotations with unique object IDs

In [ ]:
# Configuration
FILENAME = SEGMENT_NAME
NUM_FRAMES = 30  # Number of frames to load for analysis

# Camera name mapping (Waymo uses integer IDs)
CAMERA_NAMES = {1: "FRONT", 2: "FRONT_LEFT", 3: "FRONT_RIGHT", 4: "SIDE_LEFT", 5: "SIDE_RIGHT"}

# Label class mapping
LABEL_MAP = {
    1: ("Vehicle",    "#FF4444"),
    2: ("Pedestrian", "#44FF44"),
    3: ("Sign",       "#FFFF44"),
    4: ("Cyclist",    "#44FFFF")
}

# Load frames from TFRecord
def load_frames(filepath, n=30):
    """Parse N frames from a Waymo TFRecord file."""
    dataset = tf.data.TFRecordDataset(filepath, compression_type='')
    frames = []
    for record in dataset.take(n):
        frame = dataset_pb2.Frame()
        frame.ParseFromString(record.numpy())
        frames.append(frame)
    return frames

print(f"Loading {NUM_FRAMES} frames...")
frames = load_frames(FILENAME, NUM_FRAMES)
print(f"Loaded {len(frames)} frames.")

# Quick dataset summary
frame0 = frames[0]
print(f"\nDataset Summary:")
print(f"  Cameras     : {len(frame0.images)}")
print(f"  Camera names: {[CAMERA_NAMES.get(img.name) for img in frame0.images]}")
print(f"  Objects in frame 0: {sum(len(cl.labels) for cl in frame0.camera_labels)}")

---
## 4. Visualization

### 4a. Single Frame with Ground-Truth Bounding Boxes

Each bounding box is color-coded by object class:  
🔴 Vehicle &nbsp; 🟢 Pedestrian &nbsp; 🟡 Sign &nbsp; 🔵 Cyclist

In [ ]:
def draw_boxes(ax, frame, camera_index=0):
    """Draw ground-truth bounding boxes on a matplotlib axis."""
    if camera_index >= len(frame.camera_labels):
        return
    for label in frame.camera_labels[camera_index].labels:
        box = label.box
        name, color = LABEL_MAP.get(label.type, ("Unknown", "white"))
        x = box.center_x - box.length / 2
        y = box.center_y - box.width / 2
        rect = patches.Rectangle(
            (x, y), box.length, box.width,
            linewidth=2, edgecolor=color,
            facecolor=color, alpha=0.15
        )
        ax.add_patch(rect)
        ax.text(
            x + 3, y + 13, name,
            fontsize=7, color="black",
            bbox=dict(facecolor=color, alpha=0.85, pad=1, boxstyle="round,pad=0.2")
        )

# Display frame 0 — FRONT camera
img = tf.image.decode_jpeg(frames[0].images[0].image).numpy()

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(img)
draw_boxes(ax, frames[0], camera_index=0)
ax.set_title("Frame 0 — FRONT Camera — Ground Truth Detections", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

### 4b. Multi-Frame Grid — Visualizing Motion Over Time

In [ ]:
# Display first 10 frames in a grid to visualize scene progression
DISPLAY_FRAMES = 10
cols = 5
rows = math.ceil(DISPLAY_FRAMES / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, 4 * rows))
axes = axes.flatten()

for i in range(DISPLAY_FRAMES):
    img = tf.image.decode_jpeg(frames[i].images[0].image).numpy()
    axes[i].imshow(img)
    draw_boxes(axes[i], frames[i], camera_index=0)
    axes[i].set_title(f"Frame {i}", fontsize=9)
    axes[i].axis("off")

# Hide unused subplots
for j in range(DISPLAY_FRAMES, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("FRONT Camera — Frames 0 to 9 with Ground-Truth Detections", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Object Tracking

Waymo provides a **unique ID per object** that persists across all frames in a segment.  
This lets us track objects without training any model — we use the ground-truth IDs directly.

### 5a. Single Object Tracking with Motion Trail

In [ ]:
def find_best_target(frames, min_frames=4):
    """
    Find the object ID that:
    - Appears in the most frames
    - Has the largest average bounding box area (most visible)
    Returns (object_id, camera_index)
    """
    id_info = defaultdict(lambda: {"count": 0, "total_area": 0.0, "cam_idx": 0})

    for frame in frames:
        for cam_idx, cam_labels in enumerate(frame.camera_labels):
            for label in cam_labels.labels:
                oid = label.id
                area = label.box.length * label.box.width
                id_info[oid]["count"] += 1
                id_info[oid]["total_area"] += area
                id_info[oid]["cam_idx"] = cam_idx

    # Score: favour objects visible in many frames with large boxes
    scored = {
        oid: info["count"] * (info["total_area"] / info["count"])
        for oid, info in id_info.items()
        if info["count"] >= min_frames
    }

    if not scored:
        raise ValueError("No object found in 4+ frames. Try loading more frames.")

    best_id = max(scored, key=scored.get)
    cam_idx = id_info[best_id]["cam_idx"]
    print(f"Selected object: {best_id}")
    print(f"  Visible in {id_info[best_id]['count']} frames")
    print(f"  Avg box area : {id_info[best_id]['total_area'] / id_info[best_id]['count']:.0f} px²")
    return best_id, cam_idx

target_id, target_cam = find_best_target(frames)

In [ ]:
# Track single object across 5 frames with motion trail
TRACK_FRAMES = 5
fig, axes = plt.subplots(1, TRACK_FRAMES, figsize=(20, 5))

trail_x, trail_y = [], []

for i in range(TRACK_FRAMES):
    frame = frames[i]
    img = tf.image.decode_jpeg(frame.images[target_cam].image).numpy()
    ax = axes[i]
    ax.imshow(img)
    ax.set_title(f"Frame {i}", fontsize=10)
    ax.axis("off")

    for label in frame.camera_labels[target_cam].labels:
        if label.id == target_id:
            box = label.box
            cx, cy = box.center_x, box.center_y
            trail_x.append(cx)
            trail_y.append(cy)

            # Bounding box
            rect = patches.Rectangle(
                (cx - box.length/2, cy - box.width/2),
                box.length, box.width,
                linewidth=3, edgecolor="cyan", facecolor="cyan", alpha=0.15
            )
            ax.add_patch(rect)

            # Motion trail
            if len(trail_x) > 1:
                ax.plot(trail_x, trail_y, linewidth=3, color="cyan",
                        marker="o", markersize=5, alpha=0.8)

            ax.scatter(cx, cy, s=80, color="white", zorder=5)
            break

fig.suptitle("Single Object Tracking — Cyan box follows the same object across frames",
             fontsize=12)
plt.tight_layout()
plt.show()

### 5b. Multi-Object Tracking — All Objects with Unique Colors

In [ ]:
# Assign each unique object ID a consistent random color
random.seed(42)  # Fixed seed = reproducible colors
id_colors = {}
id_trajectories = defaultdict(list)

TRACK_FRAMES = 10
cols = 5
rows = math.ceil(TRACK_FRAMES / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, 4 * rows))
axes = axes.flatten()

for i in range(TRACK_FRAMES):
    frame = frames[i]
    img = tf.image.decode_jpeg(frame.images[0].image).numpy()
    ax = axes[i]
    ax.imshow(img)
    ax.set_title(f"Frame {i}", fontsize=9)
    ax.axis("off")

    for label in frame.camera_labels[0].labels:
        oid = label.id

        if oid not in id_colors:
            id_colors[oid] = (random.random(), random.random(), random.random())

        color = id_colors[oid]
        box = label.box
        cx, cy = box.center_x, box.center_y
        id_trajectories[oid].append((cx, cy))

        rect = patches.Rectangle(
            (cx - box.length/2, cy - box.width/2),
            box.length, box.width,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)

        # Draw trajectory trail
        if len(id_trajectories[oid]) > 1:
            xs = [p[0] for p in id_trajectories[oid]]
            ys = [p[1] for p in id_trajectories[oid]]
            ax.plot(xs, ys, linewidth=2, color=color, marker="o", markersize=3, alpha=0.7)

# Hide unused axes
for j in range(TRACK_FRAMES, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Multi-Object Tracking — Each color = unique object across all frames",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Trajectory Extraction

Extract the (x, y) center positions of every tracked object across all frames.  
This produces a structured table used for motion prediction modeling.

In [ ]:
# Extract trajectory records from all loaded frames
trajectory_records = []

for frame_idx, frame in enumerate(frames):
    for label in frame.camera_labels[0].labels:
        trajectory_records.append({
            "object_id"  : label.id,
            "frame"      : frame_idx,
            "class"      : LABEL_MAP.get(label.type, ("Unknown", ""))[0],
            "center_x"   : label.box.center_x,
            "center_y"   : label.box.center_y,
            "box_width"  : label.box.width,
            "box_length" : label.box.length,
        })

df = pd.DataFrame(trajectory_records)

print(f"Total records  : {len(df)}")
print(f"Unique objects : {df['object_id'].nunique()}")
print(f"Frames covered : {df['frame'].nunique()}")
print(f"\nClass distribution:")
print(df['class'].value_counts().to_string())
print(f"\nSample records:")
df.head(6)

In [ ]:
# Save trajectory data to CSV for reproducibility
CSV_PATH = "trajectories.csv"
df.to_csv(CSV_PATH, index=False)
print(f"Saved {len(df)} records to {CSV_PATH}")

# Optional: upload to your own GCS bucket
# YOUR_BUCKET = "gs://your-bucket-name"
# !gsutil cp {CSV_PATH} {YOUR_BUCKET}/trajectories.csv

---
## 7. Motion Prediction Model

**Goal:** Given an object's past positions, predict its next position.

**Approach:**
- Use a sliding window of past frames as input features
- Include velocity (dx, dy) to give the model motion context
- Baseline model: Linear Regression (fast, interpretable, strong on linear motion)

```
Input:  [x₀, y₀, x₁, y₁, x₂, y₂, vx₁, vy₁, vx₂, vy₂]
Output: [x₃, y₃]
```

In [ ]:
def build_features(df, window_size=3):
    """
    Build ML features from trajectory DataFrame.
    
    For each object, create sliding windows:
      Features: past positions + velocities
      Target  : next position
    """
    X, y = [], []

    for obj_id, group in df.groupby("object_id"):
        data = group.sort_values("frame")[["center_x", "center_y"]].values

        if len(data) <= window_size:
            continue

        for i in range(len(data) - window_size):
            past   = data[i : i + window_size]
            future = data[i + window_size]

            # Position features
            pos_features = past.flatten().tolist()

            # Velocity features (change in position between consecutive frames)
            vel_features = []
            for j in range(1, window_size):
                vel_features.extend((past[j] - past[j-1]).tolist())

            X.append(pos_features + vel_features)
            y.append(future.tolist())

    return np.array(X), np.array(y)


# Find best window size via cross-validation
print("Tuning window size...")
print(f"{'Window':>6} | {'Samples':>7} | {'Test R²':>8}")
print("-" * 30)

best_window, best_score = 2, -999
results = []

for w in range(2, 8):
    X, y = build_features(df, window_size=w)
    if len(X) < 4:
        continue

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
    m = LinearRegression().fit(X_tr, y_tr)
    score = m.score(X_te, y_te)
    results.append((w, len(X), score))
    print(f"{w:>6} | {len(X):>7} | {score:>8.4f}")

    if score > best_score:
        best_score, best_window = score, w

print(f"\nBest window size: {best_window} (R² = {best_score:.4f})")

In [ ]:
# Train final model with best window size
X, y = build_features(df, window_size=best_window)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"Final Model Results (window={best_window}):")
print(f"  Train R²  : {model.score(X_train, y_train):.4f}")
print(f"  Test R²   : {model.score(X_test, y_test):.4f}")
print(f"  Test RMSE : {rmse:.2f} pixels")

### 7b. Evaluate — Predicted vs Actual Positions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Scatter of predicted vs actual positions ---
ax = axes[0]
ax.scatter(y_test[:, 0], y_test[:, 1], s=20, alpha=0.6, label="Actual",    color="steelblue")
ax.scatter(preds[:, 0],  preds[:, 1],  s=20, alpha=0.6, label="Predicted", color="tomato",  marker="x")
ax.set_xlabel("X position (pixels)")
ax.set_ylabel("Y position (pixels)")
ax.set_title("Predicted vs Actual Positions")
ax.legend()

# --- Plot 2: Error distribution ---
errors = np.sqrt(np.sum((preds - y_test) ** 2, axis=1))  # Euclidean distance
ax2 = axes[1]
ax2.hist(errors, bins=20, color="steelblue", edgecolor="white", alpha=0.85)
ax2.axvline(np.median(errors), color="red", linestyle="--", label=f"Median: {np.median(errors):.1f}px")
ax2.set_xlabel("Prediction Error (pixels)")
ax2.set_ylabel("Count")
ax2.set_title("Error Distribution")
ax2.legend()

plt.suptitle(f"Linear Regression Motion Predictor  |  Window={best_window}  |  RMSE={rmse:.1f}px",
             fontsize=12)
plt.tight_layout()
plt.show()

---
## Next Steps

This notebook establishes the full data → visualization → tracking → prediction pipeline.

| Step | Status | Description |
|------|--------|-------------|
| Data Access | ✅ | GCS streaming via authenticated gsutil |
| Parsing | ✅ | TFRecord → protobuf frame objects |
| Visualization | ✅ | Color-coded bounding boxes, multi-frame grid |
| Object Tracking | ✅ | Ground-truth ID tracking with motion trails |
| Trajectory Extraction | ✅ | Structured DataFrame + CSV export |
| Motion Prediction | ✅ | Linear Regression baseline with RMSE evaluation |
| Deep Learning Model | 🔲 | Replace LinearRegression with LSTM/Transformer |
| Scale to Full Dataset | 🔲 | Process all 798 segments via GCS extraction pipeline |
| Object Detection Training | 🔲 | Fine-tune YOLOv8 on extracted Waymo images |
| Deployment | 🔲 | FastAPI inference endpoint + Streamlit demo |
